[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/12-proyecto-final-y-cierre/soluciones.ipynb)

# Clase 12 - Proyecto Final y Cierre (SOLUCIONES)

**Este notebook integra TODO lo aprendido en el bootcamp.**

> Este archivo contiene las soluciones completas. Intenta el notebook primero!


## Estructura del Proyecto de Data Science

```
CICLO COMPLETO:
  1. Definicion del problema
  2. Recoleccion de datos
  3. EDA
  4. Limpieza y Feature Engineering
  5. Analisis Descriptivo
  6. Modelado ML
  7. Evaluacion
  8. Comunicacion
  9. Despliegue
```


In [ ]:
# Qué hace: arma un pipeline de preprocesamiento y lo valida con varias particiones.
# Para qué sirve: evita pasos manuales inconsistentes y entrega una evaluaci?n más confiable del modelo.

# ============================================================
# BLOQUE: Importaciones completas
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, cross_val_score,
    GridSearchCV, StratifiedKFold, learning_curve
)
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
)

import joblib

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100

import warnings
warnings.filterwarnings('ignore')

print("Todas las librerias cargadas correctamente")


## Seccion 1: EDA Completo


In [ ]:
# Qué hace: carga el dataset base y deja la tabla lista para inspección o limpieza.
# Para qué sirve: asegura que el análisis parta desde una fuente común y verificable.

# ============================================================
# SOLUCION: Cargar Dataset y EDA
# ============================================================

# SOLUCION: Cargar dataset
df = pd.read_csv('../../datasets/ventas_tienda.csv')

print("=" * 50)
print("1. DIMENSIONES")
print("=" * 50)
print(f"  Filas:    {df.shape[0]}")
print(f"  Columnas: {df.shape[1]}")

print("\n2. TIPOS DE DATOS")
print(df.dtypes)

print("\n3. VALORES NULOS")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else "No hay valores nulos")

print("\n4. ESTADISTICAS DESCRIPTIVAS")
df.describe()


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# Visualizacion EDA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('EDA: Distribucion de Variables Numericas', fontsize=16, fontweight='bold')

axes[0, 0].hist(df['total_venta'], bins=50, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribucion de total_venta')
axes[0, 0].set_xlabel('Total Venta ($)')

axes[0, 1].hist(df['precio_unitario'], bins=40, color='darkorange', edgecolor='white')
axes[0, 1].set_title('Distribucion de precio_unitario')

axes[1, 0].hist(df['unidades_vendidas'], bins=30, color='green', edgecolor='white', alpha=0.8)
axes[1, 0].set_title('Distribucion de unidades_vendidas')

axes[1, 1].hist(df['descuento_pct'], bins=20, color='purple', edgecolor='white', alpha=0.8)
axes[1, 1].set_title('Distribucion de descuento_pct')

plt.tight_layout()
plt.show()


## Seccion 2: Limpieza y Feature Engineering


In [ ]:
# Qué hace: transforma texto o fechas en columnas más útiles para analizar.
# Para qué sirve: habilita filtros, agrupaciones y variables derivadas con sentido analítico.

# ============================================================
# SOLUCION: Feature Engineering
# ============================================================

df_clean = df.copy()

# Convertir fecha a datetime
df_clean['fecha'] = pd.to_datetime(df_clean['fecha'], errors='coerce')

# SOLUCION: Extraer mes
df_clean['mes'] = df_clean['fecha'].dt.month

# SOLUCION: Extraer dia de semana
df_clean['dia_semana'] = df_clean['fecha'].dt.dayofweek

# Feature fin de semana
df_clean['es_fin_de_semana'] = (df_clean['dia_semana'] >= 5).astype(int)

# Feature trimestre
df_clean['trimestre'] = df_clean['fecha'].dt.quarter

print("Feature Engineering completado")
print(f"Dataset: {df_clean.shape}")
print(df_clean[['fecha', 'mes', 'dia_semana', 'es_fin_de_semana', 'trimestre']].head(5))


In [ ]:
# Qué hace: ejecuta el paso actual del ejercicio o laboratorio.
# Para qué sirve: deja explícita la intención del bloque y ayuda a revisarlo con más criterio.

# Encoding de categoricas
df_model = pd.get_dummies(
    df_clean.drop(columns=['fecha']),
    columns=['sucursal', 'categoria'],
    drop_first=True
)

print(f"Shape final: {df_model.shape}")


## Seccion 3: Analisis Descriptivo


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# ============================================================
# SOLUCION: Analisis Descriptivo
# ============================================================

ventas_por_sucursal = (
    df_clean.groupby('sucursal')['total_venta']
    .sum().sort_values(ascending=False)
)

ventas_por_categoria = (
    df_clean.groupby('categoria')['total_venta']
    .sum().sort_values(ascending=False)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colores_barras = ['gold' if i == 0 else 'steelblue' for i in range(len(ventas_por_sucursal))]
ax1.bar(ventas_por_sucursal.index, ventas_por_sucursal.values,
        color=colores_barras, edgecolor='white')
ax1.set_title('Ventas Totales por Sucursal', fontsize=14, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

ax2.barh(ventas_por_categoria.index, ventas_por_categoria.values,
         color='darkorange', edgecolor='white')
ax2.set_title('Ventas Totales por Categoria', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("Top 3 sucursales:")
print(ventas_por_sucursal.head(3))


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# SOLUCION: Tendencia mensual

# SOLUCION: ventas_por_mes
ventas_por_mes = df_clean.groupby('mes')['total_venta'].sum()
ventas_prom_mes = df_clean.groupby('mes')['total_venta'].mean()

nombres_meses = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun',
                 'Jul', 'Ago', 'Sep', 'Oct', 'Nov', 'Dic']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

meses = ventas_por_mes.index.tolist()
ax1.bar(meses, ventas_por_mes.values, color='steelblue', edgecolor='white')
ax1.set_title('Ventas Totales por Mes', fontsize=13)
ax1.set_xticks(meses)
ax1.set_xticklabels([nombres_meses[m-1] for m in meses])

ax2.plot(meses, ventas_prom_mes.values, 'o-', color='darkorange', linewidth=2, markersize=8)
ax2.fill_between(meses, ventas_prom_mes.values, alpha=0.2, color='darkorange')
ax2.set_title('Venta Promedio por Mes', fontsize=13)
ax2.set_xticks(meses)
ax2.set_xticklabels([nombres_meses[m-1] for m in meses])

plt.tight_layout()
plt.show()


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# Heatmap de correlaciones
cols_numericas = df_clean.select_dtypes(include=np.number).columns.tolist()
cols_para_corr = [c for c in cols_numericas if c not in ['dia_semana']]
corr_matrix = df_clean[cols_para_corr].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=-1, vmax=1, center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title('Heatmap de Correlaciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

corr_con_target = corr_matrix['total_venta'].sort_values(ascending=False)
print("Correlacion con total_venta:")
print(corr_con_target)


## Seccion 4: Regresion - Predecir total_venta


In [ ]:
# Qué hace: separa variables, entrena un modelo supervisado y obtiene predicciones o scores.
# Para qué sirve: permite conectar la pregunta predictiva con una primera solución medible.

# ============================================================
# SOLUCION: Modelo de Regresion
# ============================================================

target_reg = 'total_venta'
feature_cols_reg = [c for c in df_model.columns if c != target_reg]
X_reg = df_model[feature_cols_reg]
y_reg = df_model[target_reg]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# SOLUCION: Crear y entrenar LinearRegression
reg_model = LinearRegression()
reg_model.fit(X_tr, y_tr)
print("Modelo de regresion entrenado")

# SOLUCION: Predecir
y_pred_reg = reg_model.predict(X_te)

mae  = mean_absolute_error(y_te, y_pred_reg)
rmse = np.sqrt(mean_squared_error(y_te, y_pred_reg))
r2   = r2_score(y_te, y_pred_reg)

print(f"\nMAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R2:   {r2:.4f}  ({r2*100:.1f}% variabilidad explicada)")


In [ ]:
# Qué hace: construye una visualización a partir del resumen preparado.
# Para qué sirve: permite comunicar el hallazgo central con una lectura más rápida.

# Visualizacion regresion
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y_te, y_pred_reg, alpha=0.4, color='steelblue', edgecolors='white', s=50)
min_v, max_v = min(y_te.min(), y_pred_reg.min()), max(y_te.max(), y_pred_reg.max())
ax1.plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=2, label='Perfecta')
ax1.set_xlabel('Valores Reales')
ax1.set_ylabel('Predicciones')
ax1.set_title(f'Predicciones vs Reales (R2={r2:.3f})')
ax1.legend()
ax1.grid(True, alpha=0.3)

residuos = y_te.values - y_pred_reg
ax2.hist(residuos, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax2.axvline(x=0, color='red', linewidth=2, linestyle='--')
ax2.set_xlabel('Residuo')
ax2.set_title(f'Residuos (media={residuos.mean():.2f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Seccion 5: Clasificacion - Ventas Altas vs Bajas


In [ ]:
# Qué hace: divide los datos en entrenamiento y prueba antes de modelar.
# Para qué sirve: reserva ejemplos no vistos para evaluar si el modelo generaliza.

# ============================================================
# SOLUCION: Clasificacion Binaria
# ============================================================

mediana_venta = df_model['total_venta'].median()
print(f"Mediana de total_venta: {mediana_venta:.2f}")

# SOLUCION: Crear target binario
df_model['ventas_altas'] = (df_model['total_venta'] > mediana_venta).astype(int)

print(f"Distribucion:\n{df_model['ventas_altas'].value_counts()}")
print(f"Proporcion positivos: {df_model['ventas_altas'].mean()*100:.1f}%")

feature_cols_clf = [c for c in df_model.columns
                    if c not in ['total_venta', 'ventas_altas']]

X_clf = df_model[feature_cols_clf]
y_clf = df_model['ventas_altas']

X_tr_c, X_te_c, y_tr_c, y_te_c = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

print(f"\nTrain: {X_tr_c.shape[0]}  Test: {X_te_c.shape[0]}")


In [ ]:
# Qué hace: arma un pipeline de preprocesamiento y lo valida con varias particiones.
# Para qué sirve: evita pasos manuales inconsistentes y entrega una evaluaci?n más confiable del modelo.

# ============================================================
# SOLUCION: Pipeline de Clasificacion
# ============================================================

# SOLUCION: Crear Pipeline
clf_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(max_iter=1000, random_state=42))
])

# SOLUCION: Entrenar pipeline
clf_pipeline.fit(X_tr_c, y_tr_c)
print("Clasificador entrenado")

# SOLUCION: Predecir
y_pred_clf = clf_pipeline.predict(X_te_c)

acc_clf = accuracy_score(y_te_c, y_pred_clf)
f1_clf  = f1_score(y_te_c, y_pred_clf, zero_division=0)

print(f"\nAccuracy: {acc_clf:.4f}")
print(f"F1 Score: {f1_clf:.4f}")
print("\nClassification Report:")
print(classification_report(y_te_c, y_pred_clf,
                             target_names=['Ventas Bajas', 'Ventas Altas']))

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_te_c, y_pred_clf,
    display_labels=['Ventas Bajas', 'Ventas Altas'],
    cmap='Blues', ax=ax
)
ax.set_title('Matriz de Confusion - Clasificacion de Ventas', fontsize=13)
plt.tight_layout()
plt.show()


## Seccion 6: Conclusiones de Negocio

### Hallazgos Principales

#### Analisis Descriptivo:
- La sucursal con mayores ventas se puede identificar en el grafico anterior
- La correlacion mas fuerte con `total_venta` es `precio_unitario` y `unidades_vendidas` (logico: precio x unidades = venta)
- Los meses finales del ano suelen tener mayor actividad comercial

#### Modelo de Regresion:
- R2 alto indica que `precio_unitario` y `unidades_vendidas` predicen bien `total_venta`
- El modelo captura la relacion matematica subyacente

#### Modelo de Clasificacion:
- El Pipeline con StandardScaler mejora la Logistic Regression
- F1 > 0.75 indica un buen clasificador

#### Recomendaciones:
1. Enfocar esfuerzos de marketing en la sucursal y categoria con mayor volumen
2. Usar el modelo de clasificacion para detectar transacciones de alto valor anticipadamente
3. Investigar los meses de baja venta para oportunidades de mejora


In [ ]:
# Qué hace: ejecuta el paso actual del ejercicio o laboratorio.
# Para qué sirve: deja explícita la intención del bloque y ayuda a revisarlo con más criterio.

# ============================================================
# BONUS: Guardar modelos con joblib
# ============================================================

# Guardar modelo de regresion
ruta_modelo_reg = 'modelo_regresion_ventas.pkl'
joblib.dump(reg_model, ruta_modelo_reg)
print(f"Modelo regresion guardado: {ruta_modelo_reg}")

# Guardar pipeline de clasificacion
ruta_modelo_clf = 'pipeline_clasificacion_ventas.pkl'
joblib.dump(clf_pipeline, ruta_modelo_clf)
print(f"Pipeline clasificacion guardado: {ruta_modelo_clf}")

# Verificar carga
modelo_reg_cargado = joblib.load(ruta_modelo_reg)
pipeline_clf_cargado = joblib.load(ruta_modelo_clf)

pred_prueba_reg = modelo_reg_cargado.predict(X_te[:3])
pred_prueba_clf = pipeline_clf_cargado.predict(X_te_c[:3])

print("\nModelos cargados y verificados:")
print(f"  Regresion (3 filas):      {pred_prueba_reg.round(2)}")
print(f"  Clasificacion (3 filas):  {pred_prueba_clf}")
print("\nProyecto Final completado exitosamente!")


## Cierre del Bootcamp - Completado

```
RESUMEN COMPLETO:

Clases 01-08: Fundamentos de Python y Analisis de Datos
Clases 09-12: Machine Learning con scikit-learn

HABILIDADES ADQUIRIDAS:
  [x] Analisis y limpieza de datos con pandas
  [x] Visualizaciones profesionales
  [x] Modelos predictivos (regresion y clasificacion)
  [x] Evaluacion robusta con cross-validation
  [x] Pipelines reproducibles
  [x] Guardar modelos para produccion
```

**Felicitaciones por completar el bootcamp!**
